# 02 — Train an MLP on XOR

Goal: train a tiny MLP from scratch on the XOR problem. If the loss goes down to near zero and the predictions are correct, **Phase 2 is done**.

XOR is a classic test because a *single* neuron cannot solve it — you need at least one hidden layer with a non-linearity (which is exactly what your MLP gives you).

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import matplotlib.pyplot as plt

from nanograd.engine import Value
from nanograd.nn import MLP

random.seed(1337)  # reproducibility

## Dataset — XOR

Inputs are 4 corners of the unit square. Targets are XOR: 0 if the two bits are the same, 1 if they differ.

In [ ]:
X = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]
y = [0.0, 1.0, 1.0, 0.0]

for xi, yi in zip(X, y):
    print(xi, '->', yi)

## Build the model

A 2 → 4 → 1 MLP. 2 inputs, 4 hidden neurons (with ReLU), 1 output (linear).

In [ ]:
model = MLP(2, [4, 1])
print(f'Number of parameters: {len(model.parameters())}')

## Sanity-check forward pass

Before training, the model should output something close to 0 (because biases start at 0 and weights are small).

In [ ]:
for xi in X:
    out = model(xi)
    print(xi, '->', out.data)

## Training loop

Standard recipe:
1. Forward pass on all 4 examples
2. Compute MSE loss
3. `zero_grad()` to wipe old gradients
4. `loss.backward()`
5. Update each parameter: `p.data -= lr * p.grad`

Run for ~200 steps. The loss should drop to near 0.

In [ ]:
lr = 0.05
epochs = 200

losses = []

for step in range(epochs):
    # 1. forward pass on all 4 examples
    preds = [model(xi) for xi in X]

    # 2. MSE loss = sum( (pred - target)^2 ) / N
    loss = sum(((p - yi)**2 for p, yi in zip(preds, y)), Value(0.0)) / len(X)

    # 3. zero gradients
    model.zero_grad()

    # 4. backward pass
    loss.backward()

    # 5. SGD step
    for p in model.parameters():
        p.data -= lr * p.grad

    losses.append(loss.data)
    if step % 20 == 0:
        print(f'step {step:3d}  loss = {loss.data:.6f}')

print(f'\nfinal loss = {losses[-1]:.6f}')

## Plot the loss curve

In [ ]:
plt.plot(losses)
plt.xlabel('step')
plt.ylabel('loss')
plt.title('XOR training loss')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## Final predictions

After training, the predictions should be very close to the targets `[0, 1, 1, 0]`.

In [ ]:
for xi, yi in zip(X, y):
    pred = model(xi).data
    print(f'{xi}  target={yi}  pred={pred:.4f}')

if losses[-1] < 0.01:
    print('\nPHASE 2 COMPLETE')
else:
    print('\nLoss did not converge. Try: more epochs, higher lr, larger hidden layer, or different seed.')

## TODO — your turn

Once the basic XOR works, try one or both of these:

1. **Vary the architecture**: change `[4, 1]` to `[2, 1]` (smaller) and `[8, 8, 1]` (deeper). Observe what trains and what doesn't.
2. **Try a different lr**: too high explodes, too low never converges. Find the sweet spot.
3. **Different activation**: re-build the network with `tanh` instead of `relu` (you'd need to tweak `Neuron.__call__` or add a flag). Compare convergence.

Note any observations as markdown cells below.